# Streaming & latency anatomy: TTFT, decode rate, token economics

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 09 §9.5–9.6 (with §9.2) · type: concept-lab*

**The promise:** you will decompose one model call into its two phases — TTFT and decode — consume a streamed response, and turn that decomposition into the levers that cut real *and* perceived latency, and cost.

## 🧠 Why this matters

An LLM call has two phases with different physics, and conflating them makes you optimize the wrong thing.

- **Prefill** processes your *entire input* through the model. It's parallelizable (fast per token) but still scales with input size — it sets **TTFT** (time to first token: network + queue + prefill).
- **Decode** generates output tokens *sequentially*, at a roughly constant rate, so it scales with *output* length.

So **total ≈ TTFT + output_len / decode_rate.** Streaming doesn't change that total — it changes *when the user sees the first word*. Same wall-clock, radically better experience. And the economics follow the same asymmetry: output tokens are the expensive, sequential part.

## Objectives & prereqs

**By the end you can:**
- Consume a streamed response and read `usage` at the end.
- Split a latency budget into TTFT and decode, and rank the levers that cut each.
- Set `max_tokens` per call type and explain why the growing transcript is the silent input multiplier in loops.

**Prereqs:** `09-01-decoding-knobs-live.ipynb`; `08-01` (token counting). Read book §9.5–9.6 (and §9.2, reasoning budget) alongside this.

**Runs free & offline.** `MOCK=1` simulates the SSE delta stream and the latency table locally and deterministically. The live path (`COMPANION_MOCK=0`, `ANTHROPIC_API_KEY`) streams one short real call.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import time

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default) keeps this notebook free, offline, and deterministic.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# The latest Opus-tier model; the streaming/usage shape is identical across models.
MODEL = "claude-opus-4-8"

print("MOCK mode:", MOCK, "— offline, no API key needed" if MOCK else "— LIVE, will stream a real call")

## Consume a streamed response

The Anthropic SDK makes the consumption side trivial: open `client.messages.stream(...)`, iterate `stream.text_stream` to relay deltas as they arrive, then call `stream.get_final_message()` for the full message plus `usage`.

Our mock yields the same *shape* — timed text deltas and a final usage object — so you watch tokens "arrive" offline. The function body shows the real call you'd make with `MOCK=0`.

In [ ]:
PROMPT = "Explain TTFT in two sentences."

# A small usage record type so mock and live paths return the same shape.
class Usage:
    def __init__(self, input_tokens, output_tokens):
        self.input_tokens = input_tokens
        self.output_tokens = output_tokens

    def __repr__(self):
        return f"Usage(input_tokens={self.input_tokens}, output_tokens={self.output_tokens})"


def stream_response(prompt):
    """Print deltas as they 'arrive' and return final usage (mock or live)."""
    if MOCK:
        # Canned deltas mimic the SSE stream; the tiny sleep makes arrival visible.
        deltas = ["Time ", "to ", "first ", "token ", "is ", "the ", "wait ",
                  "before ", "any ", "output ", "appears. ", "It ", "is ",
                  "dominated ", "by ", "queueing ", "and ", "prefill."]
        for d in deltas:
            print(d, end="", flush=True)
            time.sleep(0.01)  # simulate decode pacing; deterministic and tiny
        print()
        return Usage(input_tokens=14, output_tokens=len(deltas))

    import anthropic  # imported only on the live path

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    with client.messages.stream(
        model=MODEL,
        max_tokens=128,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)  # relay to the UI as it arrives
        print()
        final = stream.get_final_message()    # full message + usage once done
    return final.usage


usage = stream_response(PROMPT)
print("\nfinal usage:", usage)

## 🔮 Predict

Streaming relayed those words one chunk at a time instead of waiting for the whole answer.

**Predict:** does streaming reduce the *total* time the call takes, or only the *perceived* time (when the user first sees output)? Decide, then run the comparison.

In [ ]:
# A deterministic local model of one call: fixed TTFT, then steady decode.
TTFT_S = 0.40            # seconds before the first token (queue + prefill)
DECODE_RATE = 50.0       # tokens/sec
OUTPUT_TOKENS = 60

total = TTFT_S + OUTPUT_TOKENS / DECODE_RATE

# Buffered (non-streaming): user sees NOTHING until the whole thing is done.
perceived_buffered = total
# Streaming: user sees the FIRST token after TTFT; total wall-clock is unchanged.
perceived_streamed = TTFT_S

print(f"total wall-clock (both modes) : {total:.2f}s")
print(f"time-to-first-output, buffered: {perceived_buffered:.2f}s  (spinner the whole time)")
print(f"time-to-first-output, streamed: {perceived_streamed:.2f}s  (reads along as it writes)")
print("\nSame total. Streaming transforms PERCEIVED latency, not real latency.")

**What you just saw.** Identical total time; the only thing streaming moved is *when the first word lands*. For a chat-style product that is the difference between a 1.5-second spinner and instant feedback — every chat product streams, and yours should too. But note the honest framing: if your actual complaint is *total* latency, streaming alone won't fix it; you need the levers below.

## A tiny latency model: vary input size and output length

TTFT grows with **input** (prefill scales with prompt size); total grows with **output** (decode is sequential). Let's build a `total ≈ TTFT(input) + output / decode_rate` table and watch which dimension hurts.

In [ ]:
def ttft(input_tokens, base=0.15, per_1k=0.06):
    """Prefill is parallel but still scales with input size."""
    return base + per_1k * (input_tokens / 1000.0)


def total_latency(input_tokens, output_tokens, decode_rate=50.0):
    return ttft(input_tokens) + output_tokens / decode_rate


print(f"{'input':>8} {'output':>8} {'TTFT(s)':>9} {'total(s)':>9}")
for inp, out in [(2_000, 60), (2_000, 600), (50_000, 60), (50_000, 600)]:
    print(f"{inp:>8} {out:>8} {ttft(inp):>9.2f} {total_latency(inp, out):>9.2f}")

**What you just saw.** A 50,000-token prompt has a visibly worse TTFT than a 2,000-token one (prefill scales with input). And a 600-token answer takes far longer than a 60-token one (decode is sequential). The two dimensions are independent — which is exactly why the levers below are ranked, not interchangeable.

## Rank the levers

The decomposition hands you levers in rough order of leverage. The chapter's ordering, from biggest to smallest single-call win:

In [ ]:
levers = [
    ("Send fewer input tokens", "trim prompt / retrieve better / compact transcript — cuts TTFT *and* cost"),
    ("Cache the static prefix", "prompt caching skips prefill for the unchanged system+tools (Ch 11) — biggest TTFT win in agents"),
    ("Generate fewer output tokens", "tight max_tokens, concise-output instructions, lean schemas"),
    ("Use a faster tier", "small models decode several x faster and queue shorter — route easy steps to them"),
    ("Stream", "no change to real latency; transforms perceived latency (usually the real complaint)"),
    ("Parallelize independent calls", "asyncio.gather for fan-out instead of awaiting in series (Ch 24)"),
]
for i, (name, why) in enumerate(levers, 1):
    print(f"{i}. {name}\n     {why}")

## Token economics: output is the expensive, sequential part

Providers price input and output separately, and **output tokens typically cost several times more**. Three habits keep the bill sane. The biggest mechanical one: set `max_tokens` per call type — a classifier needs ~10 tokens, not 4,096.

In [ ]:
# Illustrative relative prices (output costs several x input). Real numbers vary by model.
PRICE_IN_PER_1K = 0.003
PRICE_OUT_PER_1K = 0.015   # ~5x input here


def call_cost(input_tokens, output_tokens):
    return (input_tokens / 1000 * PRICE_IN_PER_1K) + (output_tokens / 1000 * PRICE_OUT_PER_1K)


# Same classification task, two max_tokens settings.
lean = call_cost(input_tokens=400, output_tokens=10)     # asked for one label
bloated = call_cost(input_tokens=400, output_tokens=4096)  # left max_tokens at a global max
print(f"classifier with max_tokens=10   : ${lean:.5f}")
print(f"classifier with max_tokens=4096 : ${bloated:.5f}  (if the model rambles to the cap)")
print(f"per-call blow-up factor         : {bloated / lean:.0f}x")

### ⚠️ The silent input multiplier in agent loops

In a loop, the *growing transcript* is re-sent on every iteration — so **input** volume, not output, is the silent cost driver (recall Ch 8). Watch input tokens balloon across eight turns even though each turn's *new* content is small.

In [ ]:
transcript_tokens = 600   # initial system prompt + first user message
per_turn_new = 150        # each turn adds a bit of assistant + tool-result text

cumulative_input = 0
for turn in range(1, 9):
    cumulative_input += transcript_tokens  # the WHOLE transcript is re-sent each turn
    print(f"turn {turn}: re-sent input = {transcript_tokens:>5} tok   (cumulative billed input: {cumulative_input:>6})")
    transcript_tokens += per_turn_new
print("\nEight round-trips re-send an ever-larger transcript — prompt caching (Ch 11) is the structural fix.")

## ⚠️ Pitfall: streaming *unvalidated* structured output

Streaming has one real cost — you're rendering output you haven't finished validating. For free-form prose that's fine. For **structured output**, never stream model text straight into something that *executes or renders as markup* (Ch 41 covers the injection risks). Buffer until the JSON parses, or use the SDK's partial-JSON helpers and render fields only as they complete.

In [ ]:
import json

# Simulate JSON arriving as a stream of chunks. We must NOT act on it until it parses.
chunks = ['{"act', 'ion": "de', 'lete", "targ', 'et": "reco', 'rd-42"}']
buffer, parsed = "", None
for c in chunks:
    buffer += c
    try:
        parsed = json.loads(buffer)   # only succeeds once the whole object has arrived
        print("buffer parses — safe to act:", parsed)
        break
    except json.JSONDecodeError:
        print(f"partial (not yet valid): {buffer!r}")

assert parsed is not None, "never act on a half-streamed action object"
print("\nBuffered until valid — we never executed 'delete' on a partial parse.")

## 🎯 Senior lens

Per-call tuning is the *small* lever. In an agent the dominant term is the **number of sequential calls**: eight model round-trips means eight TTFTs *and* eight decode phases in series — and that's architectural, not a parameter. The senior moves are structural: combine steps a capable model can do in one pass, parallelize independent branches, route trivial decisions to fast models or plain code, and cache aggressively. Profile a slow agent the way you'd profile any distributed system — find the critical path first (Ch 23's traces make it visible), *then* tune parameters.

And the single biggest per-step cost/latency lever is the **reasoning thinking budget** (§9.2): high-effort thinking is billed like output and adds latency the user feels, so spend it only on genuinely hard steps (planning, an ambiguous tool choice) and dial it down for classification, extraction, routing, and formatting.

## Recap

- **A call has two phases:** prefill (parallel, scales with *input*) sets TTFT; decode (sequential, ~constant per token) scales with *output*. `total ≈ TTFT + output_len / decode_rate`.
- **Streaming changes perceived, not real, latency** — but that's usually the actual complaint.
- **Levers, ranked:** fewer input tokens → cache the prefix → fewer output tokens → faster tier → stream → parallelize.
- **Output tokens cost several x input;** set `max_tokens` per call type. In loops, the growing transcript is the silent *input* multiplier.
- **Never stream unvalidated structured output** into anything that executes or renders — buffer until it parses.
- **In agents, the dominant cost is the number of sequential calls** — and the thinking budget is the biggest per-step lever; spend both deliberately.

## Exercises

Each one *changes a knob* and asks you to predict first.

1. **Crossover point.** Using `total_latency`, find the output length at which decode time equals TTFT for a 10,000-token input. Predict whether it's above or below 30 tokens, then compute it.
2. **Cache the prefix.** Model prompt caching by setting prefill on the *unchanged* portion to ~0 in `ttft`. Predict the TTFT improvement for a 50,000-token mostly-static system prompt, then show it.
3. **Right-size `max_tokens`.** For a router that must emit one of four labels, pick a `max_tokens` and justify it; then compute the per-call cost vs. leaving it at 4096. Predict the blow-up factor before you compute it.

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **End of Ch 09's labs.** Next chapter: [`../10-prompt-engineering/`](../10-prompt-engineering/) — prompts become versioned, tested engineering artifacts.
- **Book:** §9.5 (streaming & economics), §9.6 (latency anatomy), §9.2 (reasoning budget); the streamed FastAPI endpoint arrives in §38, critical-path profiling in §23, caching/batching in §11.
- **Blueprint this feeds:** [`blueprints/llm-gateway/`](../../blueprints/llm-gateway/) — which owns streaming, retries, and usage logging (built from Ch 11).
- **Capstone:** the streamed endpoints and per-call usage logging the capstone encodes (Ch 38).